---RAG pipeline---


In [2]:
## data ingestion -> vector db pipeline
import os
from langchain_community.document_loaders import PyPDFLoader ,PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path



c:\Users\chait\OneDrive\Documents\Generative-Ai\Generative-AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
print(5)

5


In [3]:
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print(f"found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nprocessing: {pdf_file.name}")
        try:
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")

found 4 PDF files to process

processing: Lecture-1-2-heterogeneous-computing.pdf
loaded 10 pages

processing: Lecture-1-3-portability-scalability.pdf
loaded 11 pages

processing: Lecture-2-1-cuda-thrust-libs.pdf
loaded 12 pages

processing: Lecture-2-2-cuda-data-allocation-API.pdf
loaded 13 pages
Total documents loaded: 46


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 15 for PowerPoint', 'creationdate': '2016-04-01T20:45:45-05:00', 'source': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'file_path': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Module 01 - Course Introduction', 'author': 'Cook, Colleen N', 'subject': '', 'keywords': '', 'moddate': '2016-04-01T20:45:47-05:00', 'trapped': '', 'modDate': "D:20160401204547-05'00'", 'creationDate': "D:20160401204545-05'00'", 'page': 0, 'source_file': 'Lecture-1-2-heterogeneous-computing.pdf', 'file_type': 'pdf'}, page_content='Introduction to Heterogeneous Parallel Computing\nLecture 1.2 – Course Introduction\nAccelerated Computing\nGPU Teaching Kit'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 15 for PowerPoint', 'creationdate': '2016-04-01T20:45:45-05:00', 'source': '..\\data\\pdf\\Lecture-1-2-heterog

In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ","",","]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n example chunk: ")
        print(f"content: {split_docs[1].page_content} ")
        print(f"Metadata: {split_docs[1].metadata}")

        
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

split 46 into 46 chunks

 example chunk: 
content: 2
Objectives
– To learn the major differences between latency devices (CPU cores) 
and throughput devices (GPU cores)
– To understand why winning applications increasingly use both types 
of devices 
Metadata: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 15 for PowerPoint', 'creationdate': '2016-04-01T20:45:45-05:00', 'source': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'file_path': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Module 01 - Course Introduction', 'author': 'Cook, Colleen N', 'subject': '', 'keywords': '', 'moddate': '2016-04-01T20:45:47-05:00', 'trapped': '', 'modDate': "D:20160401204547-05'00'", 'creationDate': "D:20160401204545-05'00'", 'page': 1, 'source_file': 'Lecture-1-2-heterogeneous-computing.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 15 for PowerPoint', 'creationdate': '2016-04-01T20:45:45-05:00', 'source': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'file_path': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Module 01 - Course Introduction', 'author': 'Cook, Colleen N', 'subject': '', 'keywords': '', 'moddate': '2016-04-01T20:45:47-05:00', 'trapped': '', 'modDate': "D:20160401204547-05'00'", 'creationDate': "D:20160401204545-05'00'", 'page': 0, 'source_file': 'Lecture-1-2-heterogeneous-computing.pdf', 'file_type': 'pdf'}, page_content='Introduction to Heterogeneous Parallel Computing\nLecture 1.2 – Course Introduction\nAccelerated Computing\nGPU Teaching Kit'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 15 for PowerPoint', 'creationdate': '2016-04-01T20:45:45-05:00', 'source': '..\\data\\pdf\\Lecture-1-2-heterog

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


Vector Store

In [17]:
import chromadb
class VectorStore:
    def __init__(self,collection_name: str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        ##initialize chroma client and collection
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document  embedding for RAG"}
            )
            print(f"Vector store initialized, collection :{self.collection_name}")
            print(f"Exisiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents should match the no of embeddings")
        
        print(f"adding {len(documents)} to vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to the vector store")
            print(f"Total documents in collection {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to vector store {e}")
            raise

    
vector_store=VectorStore()
vector_store

Vector store initialized, collection :pdf_documents
Exisiting documents in collection: 0


In [18]:
### converting text to embeddings
texts=[doc.page_content for doc in chunks]


embeddings=embedding_manager.generate_embeddings(texts=texts)

vector_store.add_documents(chunks,embeddings)

Generating embeddings for 46 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.30it/s]


Generated embeddings with shape: (46, 384)
adding 46 to vector store
successfully added 46 documents to the vector store
Total documents in collection 46


Retrieval pipeline from vector store

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("scalability")

Retrieving documents for query: 'scalability'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 108.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc1791af74_11',
  'content': '2\nObjectives\n– To understand the importance and nature of scalability and \nportability in parallel programming',
  'metadata': {'moddate': '2016-04-01T20:46:19-05:00',
   'trapped': '',
   'content_length': 110,
   'keywords': '',
   'format': 'PDF 1.5',
   'author': 'Cook, Colleen N',
   'modDate': "D:20160401204619-05'00'",
   'creationDate': "D:20160401204617-05'00'",
   'doc_index': 11,
   'creator': 'Acrobat PDFMaker 15 for PowerPoint',
   'source': '..\\data\\pdf\\Lecture-1-3-portability-scalability.pdf',
   'total_pages': 11,
   'producer': 'Adobe PDF Library 15.0',
   'subject': '',
   'title': 'Module 01 - Course Introduction',
   'file_path': '..\\data\\pdf\\Lecture-1-3-portability-scalability.pdf',
   'source_file': 'Lecture-1-3-portability-scalability.pdf',
   'page': 1,
   'creationdate': '2016-04-01T20:46:17-05:00',
   'file_type': 'pdf'},
  'similarity_score': 0.195673406124115,
  'distance': 0.804326593875885,
  'rank': 1}]

In [28]:
rag_retriever.retrieve("Latency Oriented Design")

Retrieving documents for query: 'Latency Oriented Design'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc0ee272d8_4',
  'content': '5\nCPUs: Latency Oriented Design \n5\n–\nPowerful ALU\n–\nReduced operation latency\n–\nLarge caches\n–\nConvert long latency memory \naccesses to short latency cache \naccesses\n–\nSophisticated control\n–\nBranch prediction for reduced \nbranch latency\n–\nData forwarding for reduced data \nlatency\nCache\nALU\nControl\nALU\nALU\nALU\nDRAM\nCPU',
  'metadata': {'modDate': "D:20160401204547-05'00'",
   'author': 'Cook, Colleen N',
   'creator': 'Acrobat PDFMaker 15 for PowerPoint',
   'subject': '',
   'title': 'Module 01 - Course Introduction',
   'keywords': '',
   'creationdate': '2016-04-01T20:45:45-05:00',
   'producer': 'Adobe PDF Library 15.0',
   'file_type': 'pdf',
   'file_path': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf',
   'source': '..\\data\\pdf\\Lecture-1-2-heterogeneous-computing.pdf',
   'content_length': 320,
   'doc_index': 4,
   'total_pages': 10,
   'trapped': '',
   'format': 'PDF 1.5',
   'creationDate': "D:2

In [35]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        print("No relevant context found")

    ## generate the answer using groq llm
    prompt="""Use the following context to answer the question consicely
    Context:{context}
    Question:{query}
    Answer:
    """

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [39]:
answer=rag_simple("what my name",rag_retriever,llm)
print(answer)


Retrieving documents for query: 'what my name'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.45it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found
I don't have that information.


In [41]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("explain more on scalability", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'explain more on scalability'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


Answer: **Scalability in Parallel Programming:**

Scalability refers to the ability of a parallel program to efficiently utilize an increasing number of processing units (e.g., cores, nodes) without a significant decrease in performance. It involves designing the program to scale up or down as needed, handling larger or smaller datasets, and adapting to varying system configurations.

Key aspects of scalability:

1. **Linear scalability**: The program's performance improves proportionally with the number of processing units.
2. **Efficient communication**: Scalable programs minimize communication overhead between processing units.
3. **Load balancing**: The program distributes the workload evenly across processing units to maximize utilization.
4. **Adaptability**: The program can adjust to changes in system configuration, such as adding or removing processing units.

Achieving scalability requires careful design, including:

1. **Modular code**: Breaking down the program into smaller,

In [43]:
result = rag_advanced("latency oriented design", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'latency oriented design'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: Latency Oriented Design is a CPU design approach that focuses on reducing the time it takes for the CPU to perform a single operation, typically used in applications where low latency is critical.
Sources: [{'source': 'Lecture-1-2-heterogeneous-computing.pdf', 'page': 4, 'score': 0.20952552556991577, 'preview': '5\nCPUs: Latency Oriented Design \n5\n–\nPowerful ALU\n–\nReduced operation latency\n–\nLarge caches\n–\nConvert long latency memory \naccesses to short latency cache \naccesses\n–\nSophisticated control\n–\nBranch prediction for reduced \nbranch latency\n–\nData forwarding for reduced data \nlatency\nCache\nALU\nControl\n...'}, {'source': 'Lecture-1-2-heterogeneous-computing.pdf', 'page': 1, 'score': 0.11288660764694214, 'preview': '2\nObjectives\n– To learn the major differences between latency devices (CPU cores) \nand throughput devices (GPU cores)\n– To understand why winning applications increasingly use both types \nof devices...'}]
Confidence: 0.2095255255699157